<a href="https://www.kaggle.com/code/secretiveplotter1863/titanicml?scriptVersionId=331851796" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/notebooks/alihassankamal/titanic-data/__results__.html
/kaggle/input/notebooks/alihassankamal/titanic-data/__notebook__.ipynb
/kaggle/input/notebooks/alihassankamal/titanic-data/__output__.json
/kaggle/input/notebooks/alihassankamal/titanic-data/custom.css
/kaggle/input/notebooks/alihassankamal/titanic-data/__results___files/__results___11_0.png
/kaggle/input/notebooks/alihassankamal/titanic-data/__results___files/__results___14_0.png
/kaggle/input/notebooks/alihassankamal/titanic-data/__results___files/__results___12_0.png
/kaggle/input/notebooks/alihassankamal/titanic-data/__results___files/__results___13_0.png
/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
from sklearn.datasets import fetch_openml
import pandas as pd
titanic=fetch_openml('titanic', version=1, as_frame=True)
df=titanic.frame
print(df.head())

   pclass survived                                             name     sex  \
0       1        1                    Allen, Miss. Elisabeth Walton  female   
1       1        1                   Allison, Master. Hudson Trevor    male   
2       1        0                     Allison, Miss. Helen Loraine  female   
3       1        0             Allison, Mr. Hudson Joshua Creighton    male   
4       1        0  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female   

       age  sibsp  parch  ticket      fare    cabin embarked boat   body  \
0  29.0000      0      0   24160  211.3375       B5        S    2    NaN   
1   0.9167      1      2  113781  151.5500  C22 C26        S   11    NaN   
2   2.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3  30.0000      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4  25.0000      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest  
0                     St Louis,

In [3]:
print(df.isnull().sum)

<bound method DataFrame.sum of       pclass  survived   name    sex    age  sibsp  parch  ticket   fare  \
0      False     False  False  False  False  False  False   False  False   
1      False     False  False  False  False  False  False   False  False   
2      False     False  False  False  False  False  False   False  False   
3      False     False  False  False  False  False  False   False  False   
4      False     False  False  False  False  False  False   False  False   
...      ...       ...    ...    ...    ...    ...    ...     ...    ...   
1304   False     False  False  False  False  False  False   False  False   
1305   False     False  False  False   True  False  False   False  False   
1306   False     False  False  False  False  False  False   False  False   
1307   False     False  False  False  False  False  False   False  False   
1308   False     False  False  False  False  False  False   False  False   

      cabin  embarked   boat   body  home.dest  
0     F

In [4]:
age=df['age']
median=df['age'].median()
df['age']=df['age'].fillna(median)
print(df['age'])

0       29.0000
1        0.9167
2        2.0000
3       30.0000
4       25.0000
         ...   
1304    14.5000
1305    28.0000
1306    26.5000
1307    27.0000
1308    29.0000
Name: age, Length: 1309, dtype: float64


In [5]:
df_encoded = pd.get_dummies(df, columns=['sex', 'embarked'])
df_clean = df_encoded.drop(columns=['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest'])
print(df_clean.head())
X=df_clean.drop(columns=['survived'])

y=df_clean['survived']
y=y.values.astype(int)
# Fill missing ages with the median age
X['age'] = X['age'].fillna(X['age'].median())

# Fill missing fares with the median fare
X['fare'] = X['fare'].fillna(X['fare'].median())
X=X.astype(float)
# Double-check that everything is now 0
print(X.isnull().sum())
print(y)

   pclass survived      age  sibsp  parch      fare  sex_female  sex_male  \
0       1        1  29.0000      0      0  211.3375        True     False   
1       1        1   0.9167      1      2  151.5500       False      True   
2       1        0   2.0000      1      2  151.5500        True     False   
3       1        0  30.0000      1      2  151.5500       False      True   
4       1        0  25.0000      1      2  151.5500        True     False   

   embarked_C  embarked_Q  embarked_S  
0       False       False        True  
1       False       False        True  
2       False       False        True  
3       False       False        True  
4       False       False        True  
pclass        0
age           0
sibsp         0
parch         0
fare          0
sex_female    0
sex_male      0
embarked_C    0
embarked_Q    0
embarked_S    0
dtype: int64
[1 1 0 ... 0 0 0]


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
scaler= StandardScaler()

X_scaled=scaler.fit_transform(X)
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.2,random_state=42)

In [7]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras import Sequential

model=tf.keras.Sequential(
    [Dense(16,activation='relu',input_shape=(X_train.shape[1],)),
     Dense(1, activation='sigmoid')]
)
model.compile(loss='binary_crossentropy',
              metrics=['accuracy'],
              optimizer='adam')
history=model.fit(
    X_train,
    y_train,
    epochs=20
)

2026-07-01 14:50:33.305246: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782917433.634853      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782917433.738445      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782917434.484946      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782917434.484989      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1782917434.484993      16 computation_placer.cc:177] computation placer alr

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-07-01 14:50:51.300181: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


33/33 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5864 - loss: 0.8832
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6294 - loss: 0.7727
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6371 - loss: 0.6926
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6457 - loss: 0.6326
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6667 - loss: 0.5863
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6934 - loss: 0.5520
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7364 - loss: 0.5249
Epoch 8/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7650 - loss: 0.5052
Epoch 9/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7870 - loss: 0.4904
Epoch 10/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7985 - loss: 0.4791
Epoch 11/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8004 - loss: 0.4701
Epoch 12/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7956 - loss: 0.4637
